# Task 3: Supervised Neural PCFG

From CKY to Neural Grammar Induction · Penn Treebank, Partition A

## Settings

Set `train_supervised_model = True` to train Task 3 from scratch. With `False`, the notebook loads the saved model from `model_dir` or downloads it from the project folder.

In [1]:
from pathlib import Path

seed = 42
train_compound_model = False
train_supervised_model = False
download_models = True
model_dir = Path("checkpoints")
results_dir = Path("results")

In [2]:
import hashlib
import importlib.util
import json
import random
import subprocess
import sys
import zipfile
from collections import Counter, defaultdict
from datetime import datetime
from importlib.metadata import version
from math import log

packages = {
    "nltk": "nltk==3.9.1",
    "torch_struct": "torch-struct==0.5",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
}
if download_models:
    packages["gdown"] = "gdown==5.2.0"

missing = [package for module, package in packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])

import matplotlib.pyplot as plt
import nltk
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from nltk.corpus import treebank
from nltk.tree import Tree
from torch import nn
from torch.nn import functional as F
from torch_struct import SentCFG

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

run_dir = results_dir / datetime.now().strftime("%Y%m%d_%H%M%S_%f")
run_dir.mkdir(parents=True)
new_model_dir = run_dir / "trained_checkpoints"
new_model_dir.mkdir()

In [3]:
saved_models = {
    "task2": (
        "task2_best_epoch1_saved.pt",
        "1yDXrNZwlQzL2ifp_vFX9xEn0WlC8G4vL",
        "237073d7409caf1e9093b7db816f6686a9f87c15bf1b9b17c975b39b0fc1c87e",
    ),
    "task3": (
        "task3_epoch15.pt",
        "1TSyhTKk_6pZg8I6Lz2jJtEObzL6RwpCU",
        "73929d6170a1c937798fd628629e495dd11acc4e54f7ed1d709ecf0690d07352",
    ),
}


def file_hash(path):
    with path.open("rb") as file:
        return hashlib.file_digest(file, "sha256").hexdigest()


def checkpoint_path(task):
    filename, drive_id, expected_hash = saved_models[task]
    path = model_dir / filename

    if not path.exists():
        if not download_models:
            raise FileNotFoundError(f"Expected checkpoint at {path}")
        import gdown

        model_dir.mkdir(parents=True, exist_ok=True)
        temporary_file = path.with_suffix(".part")
        try:
            gdown.download(id=drive_id, output=str(temporary_file), use_cookies=False)
            if not temporary_file.exists() or file_hash(temporary_file) != expected_hash:
                raise ValueError(f"The download for {filename} did not match the saved model.")
            temporary_file.replace(path)
        finally:
            temporary_file.unlink(missing_ok=True)

    if download_models and file_hash(path) != expected_hash:
        raise ValueError(f"{path} differs from the shared checkpoint.")
    return path


def save_json(filename, value):
    (run_dir / filename).write_text(json.dumps(value, indent=2, ensure_ascii=False),
                                   encoding="utf-8")


package_names = ["numpy", "nltk", "torch", "torch-struct", "pandas", "matplotlib"]
if download_models:
    package_names.append("gdown")
package_versions = {package: version(package) for package in package_names}
run_info = {
    "seed": seed,
    "device": str(device),
    "python": sys.version.split()[0],
    "packages": package_versions,
    "checkpoints": {},
    "task2_mode": "train" if train_compound_model else "checkpoint",
    "task3_mode": "train" if train_supervised_model else "checkpoint",
    "evaluation": "Unique unlabeled spans after punctuation projection; no full-sentence or single-word spans.",
}


def record_checkpoint(task, path, checkpoint):
    run_info["checkpoints"][task] = {
        "filename": path.name,
        "epoch": checkpoint["epoch"],
        "sha256": file_hash(path),
        "validation_loss": checkpoint.get("best_validation_loss"),
    }

## Data and split

I shuffle the 3,914 trees with seed 42, take the first 1,305 for Partition A, and shuffle that partition again before splitting it. I remove empty categories and strip functional suffixes such as NP-SBJ. The words, case and punctuation stay in the model input.

In [4]:
def clean_tree(node):
    if isinstance(node, str):
        return node
    if node.label() == "-NONE-":
        return None

    children = [clean_tree(child) for child in node]
    children = [child for child in children if child is not None]
    if not children:
        return None

    label = node.label()
    if not label.startswith("-"):
        label = label.split("-", 1)[0].split("=", 1)[0]
    return Tree(label, children)


try:
    all_trees = list(treebank.parsed_sents())
except LookupError:
    nltk.download("treebank", quiet=True)
    all_trees = list(treebank.parsed_sents())

assert len(all_trees) == 3914
indices = np.random.RandomState(seed).permutation(len(all_trees))
partitions = {
    "A": indices[:1305].tolist(),
    "B": indices[1305:2610].tolist(),
    "C": indices[2610:].tolist(),
}
partition_a = np.random.RandomState(seed).permutation(partitions["A"])
train_end = int(0.8 * len(partition_a))
validation_end = train_end + int(0.1 * len(partition_a))
split_indices = {
    "train": partition_a[:train_end].tolist(),
    "validation": partition_a[train_end:validation_end].tolist(),
    "test": partition_a[validation_end:].tolist(),
}
trees_by_split = {
    split: [clean_tree(all_trees[index]) for index in tree_indices]
    for split, tree_indices in split_indices.items()
}
train_trees = trees_by_split["train"]
validation_trees = trees_by_split["validation"]
test_trees = trees_by_split["test"]

save_json("partition_indices.json", {"partitions": partitions, "partition_a_splits": split_indices})
split_text = json.dumps(split_indices, sort_keys=True, separators=(",", ":"))
run_info["split_sha256"] = hashlib.sha256(split_text.encode()).hexdigest()
display(pd.DataFrame([
    {"Split": split.title(), "Trees": len(trees)} for split, trees in trees_by_split.items()
]))

,Split,Trees
0,Train,1044
1,Validation,130
2,Test,131


In [5]:
def replace_rare_words(trees, cutoff):
    word_counts = Counter(word for tree in trees for word in tree.leaves())
    changed_trees = []
    for tree in trees:
        changed = tree.copy(deep=True)
        for position in changed.treepositions("leaves"):
            if word_counts[changed[position]] <= cutoff:
                changed[position] = "<UNK>"
        changed_trees.append(changed)
    return changed_trees


def to_cnf(tree, markov_order):
    binary_tree = tree.copy(deep=True)
    binary_tree.collapse_unary(collapsePOS=True, collapseRoot=True)
    binary_tree.chomsky_normal_form(factor="right", horzMarkov=markov_order)
    return binary_tree

## How I score the parses

A bracket records where a phrase starts and ends. Labeled F1 also checks its category, such as NP. The original Task 1 scores keep punctuation. For the comparison across all three tasks, I remove punctuation using the gold POS tags, count each remaining span once, and exclude single words and the full sentence. A sentence with fewer than two content words is skipped. A failed parse on an eligible sentence still counts against recall.

In [6]:
punctuation_tags = {".", ",", ":", "``", "''", "-LRB-", "-RRB-"}


def brackets(tree, labeled=False):
    spans = Counter()

    def visit(node, start, is_root=False):
        if isinstance(node, str):
            return start + 1
        end = start
        for child in node:
            end = visit(child, end)
        if not is_root and any(isinstance(child, Tree) for child in node):
            span = (start, end, node.label()) if labeled else (start, end)
            spans[span] += 1
        return end

    if tree is not None:
        visit(tree, 0, is_root=True)
    return spans


def content_length(tree):
    return sum(tag not in punctuation_tags for _, tag in tree.pos())


def project_spans(spans, prefix):
    # prefix[i] counts the content words before original position i.
    projected = set()
    for start, end in spans:
        left, right = prefix[start], prefix[end]
        if 2 <= right - left < prefix[-1]:
            projected.add((left, right))
    return projected


def score_sentence(split, index, gold_tree, prediction, predicted_spans=None, one_sided=None):
    words = gold_tree.leaves()
    gold_tags = gold_tree.pos()
    prefix = [0]
    for _, tag in gold_tags:
        prefix.append(prefix[-1] + int(tag not in punctuation_tags))

    eligible = prefix[-1] >= 2
    if prediction is not None:
        assert prediction.leaves() == words

    gold = project_spans(brackets(gold_tree), prefix)
    predicted = project_spans(
        brackets(prediction) if predicted_spans is None else predicted_spans, prefix
    )
    matches = gold & predicted
    span_count = len(gold) + len(predicted)
    sentence_f1 = 2 * len(matches) / span_count if span_count else 1.0

    return {
        "position": index, "corpus_index": split_indices[split][index],
        "tokens": words,
        "scored_words": [word for word, tag in gold_tags if tag not in punctuation_tags],
        "eligible": eligible, "complete": prediction is not None, "one_sided": one_sided,
        "matching": len(matches), "predicted_count": len(predicted), "gold_count": len(gold),
        "sentence_f1": sentence_f1 if eligible else None,
        "gold_spans": sorted(gold), "predicted_spans": sorted(predicted),
        "matching_spans": sorted(matches),
        "extra_spans": sorted(predicted - gold), "missed_spans": sorted(gold - predicted),
        "gold_tree": gold_tree.pformat(margin=10**9),
        "predicted_tree": prediction.pformat(margin=10**9) if prediction is not None else None,
    }


def bracket_scores(matches, predicted, gold):
    precision = matches / predicted if predicted else 0.0
    recall = matches / gold if gold else 0.0
    f1 = 2 * matches / (predicted + gold) if predicted + gold else 0.0
    return precision, recall, f1


def save_evaluation(name, records):
    scored = [row for row in records if row["eligible"]]
    matches = sum(row["matching"] for row in scored)
    predicted = sum(row["predicted_count"] for row in scored)
    gold = sum(row["gold_count"] for row in scored)
    precision, recall, corpus_f1 = bracket_scores(matches, predicted, gold)
    diagnosed = [row["one_sided"] for row in scored if row["one_sided"] is not None]
    summary = {
        "sentences": len(records), "scored": len(scored),
        "skipped": len(records) - len(scored),
        "complete": sum(row["complete"] for row in scored),
        "matching_spans": matches, "predicted_spans": predicted, "gold_spans": gold,
        "precision": precision, "recall": recall, "corpus_f1": corpus_f1,
        "sentence_f1": sum(row["sentence_f1"] for row in scored) / len(scored),
        "one_sided": sum(diagnosed), "long_sentences_diagnosed": len(diagnosed),
    }
    save_json(f"{name}_metrics.json", summary)
    with (run_dir / f"{name}_predictions.jsonl").open("w", encoding="utf-8") as file:
        for row in records:
            file.write(json.dumps(row, ensure_ascii=False) + "\n")
    return summary


def score_row(name, result):
    return {
        "Model": name,
        "Scored": result["scored"],
        "Complete parses": result["complete"],
        "Precision (%)": 100 * result["precision"],
        "Recall (%)": 100 * result["recall"],
        "Sentence F1 (%)": 100 * result["sentence_f1"],
        "Corpus F1 (%)": 100 * result["corpus_f1"],
    }


def show_examples(records, task):
    examples = [row for row in records if row["eligible"] and row["gold_count"]
                and 5 <= len(row["tokens"]) <= 25]
    if len(examples) < 2:
        return
    better = max(examples, key=lambda row: row["sentence_f1"])
    worse = min((row for row in examples if row is not better),
                key=lambda row: row["sentence_f1"])

    for name, row in [("High F1", better), ("Low F1", worse)]:
        print(f"{name} · test sentence {row['position']} · F1 {row['sentence_f1']:.1%}")
        print(" ".join(row["tokens"]))
        print("Gold:     ", Tree.fromstring(row["gold_tree"]).pformat(margin=95))
        predicted = row["predicted_tree"]
        print("Predicted:", Tree.fromstring(predicted).pformat(margin=95) if predicted else "No parse")
        words = row["scored_words"]
        for label, key in [("Matched", "matching_spans"), ("Missed", "missed_spans"),
                           ("Extra", "extra_spans")]:
            phrases = [" ".join(words[i:j]) for i, j in row[key]]
            print(f"{label}: {', '.join(phrases) or '—'}")
        print()

    save_json(f"{task}_examples.json", [better, worse])

## Task 3: Supervised neural PCFG

Here the training trees already give the category labels and the rule used at each node. I use the same embedding and MLP design as Task 2 without z. The labels and allowed binary rules come from the gold training trees, using Markov order 1 and the singleton word cutoff.

In [7]:
def encode_words(words, vocabulary):
    return [vocabulary.get(word, vocabulary["<UNK>"]) for word in words]


def rule_mlp(input_size, output_size):
    return nn.Sequential(nn.Linear(input_size, 256), nn.ReLU(), nn.Linear(256, output_size))


rare_word_trees = {1: replace_rare_words(train_trees, cutoff=1)}

In [8]:
supervised_trees = [to_cnf(tree, markov_order=1) for tree in rare_word_trees[1]]
observed_rules = set()
preterminals = set()
for tree in supervised_trees:
    for rule in tree.productions():
        parent = rule.lhs().symbol()
        if len(rule.rhs()) == 2:
            left, right = (child.symbol() for child in rule.rhs())
            observed_rules.add((parent, left, right))
        else:
            preterminals.add(parent)

nonterminal_labels = sorted({parent for parent, _, _ in observed_rules})
preterminal_labels = sorted(preterminals)
nonterminal_id = {label: i for i, label in enumerate(nonterminal_labels)}
preterminal_id = {label: i for i, label in enumerate(preterminal_labels)}
root_labels = sorted({tree.label() for tree in supervised_trees})
root_id = {label: i for i, label in enumerate(root_labels)}
child_pairs = sorted({(left, right) for _, left, right in observed_rules})
pair_id = {pair: i for i, pair in enumerate(child_pairs)}

words = {word for tree in supervised_trees for word in tree.leaves()}
word_list = ["<UNK>"] + sorted(words - {"<UNK>"})
supervised_vocab = {word: i for i, word in enumerate(word_list)}

allowed_pairs = torch.zeros(len(nonterminal_labels), len(child_pairs), dtype=torch.bool)
for parent, left, right in observed_rules:
    allowed_pairs[nonterminal_id[parent], pair_id[left, right]] = True

display(pd.DataFrame({
    "Training trees": [len(supervised_trees)],
    "Nonterminals": [len(nonterminal_labels)],
    "Preterminals": [len(preterminal_labels)],
    "Binary rules": [len(observed_rules)],
    "Root labels": [len(root_labels)],
    "Word IDs": [len(word_list)],
}))

,Training trees,Nonterminals,Preterminals,Binary rules,Root labels,Word IDs
0,1044,377,116,2402,8,2294


### Score a gold tree

I add the log probabilities of its root, binary rules and word emissions. With the labels and splits already given, the training loss is the negative score of that one derivation.

In [9]:
class SupervisedGrammar(nn.Module):
    def __init__(self):
        super().__init__()
        self.nt_embedding = nn.Embedding(len(nonterminal_labels), 256)
        self.pt_embedding = nn.Embedding(len(preterminal_labels), 256)
        self.root_embedding = nn.Parameter(torch.randn(256) * 0.02)
        self.root_mlp = rule_mlp(256, len(root_labels))
        self.rule_mlp = rule_mlp(256, len(child_pairs))
        self.word_mlp = rule_mlp(256, len(word_list))
        self.register_buffer("allowed", allowed_pairs.clone())

    def forward(self):
        root_scores = F.log_softmax(self.root_mlp(self.root_embedding), dim=-1)
        binary = self.rule_mlp(self.nt_embedding.weight)
        binary = binary.masked_fill(~self.allowed, float("-inf"))
        rule_scores = F.log_softmax(binary, dim=-1)
        word_scores = F.log_softmax(self.word_mlp(self.pt_embedding.weight), dim=-1)
        return root_scores, rule_scores, word_scores


def gold_tree_score(tree, root_scores, rule_scores, word_scores):
    def node_score(node):
        label = node.label()
        if isinstance(node[0], str):
            return word_scores[preterminal_id[label], supervised_vocab[node[0]]]

        left, right = node
        pair = pair_id[left.label(), right.label()]
        return rule_scores[nonterminal_id[label], pair] + node_score(left) + node_score(right)

    return root_scores[root_id[tree.label()]] + node_score(tree)


torch.manual_seed(seed)
supervised_grammar = SupervisedGrammar().to(device)
label_mappings = {
    "nt_labels": nonterminal_labels, "pt_labels": preterminal_labels,
    "rhs_pairs": child_pairs, "root_labels": root_labels, "vocab": word_list,
}

In [10]:
def train_supervised(epochs=15, batch_size=8):
    optimizer = torch.optim.Adam(supervised_grammar.parameters(), lr=1e-3, betas=(0.75, 0.999))
    for epoch in range(1, epochs + 1):
        supervised_grammar.train()
        if epoch == 1:
            order = list(range(len(supervised_trees)))
            random.Random(seed).shuffle(order)
        else:
            generator = torch.Generator().manual_seed(seed + epoch)
            order = torch.randperm(len(supervised_trees), generator=generator).tolist()

        loss_sum = 0.0
        for start in range(0, len(order), batch_size):
            batch = order[start:start + batch_size]
            scores = supervised_grammar()
            loss = torch.stack([-gold_tree_score(supervised_trees[i], *scores) for i in batch]).mean()
            if not torch.isfinite(loss):
                raise ValueError(f"Invalid Task 3 loss in epoch {epoch}")
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(supervised_grammar.parameters(), 3.0)
            optimizer.step()
            loss_sum += loss.item() * len(batch)
        print(f"Epoch {epoch:2d}: loss per tree {loss_sum / len(order):.3f}")

    path = new_model_dir / "task3_epoch15.pt"
    torch.save({
        "epoch": epochs, "model_state": supervised_grammar.state_dict(),
        "optimizer_state": optimizer.state_dict(), **label_mappings,
    }, path)
    return path


task3_path = train_supervised() if train_supervised_model else checkpoint_path("task3")
checkpoint = torch.load(task3_path, map_location="cpu", weights_only=True)
if checkpoint["epoch"] != 15:
    raise ValueError("Task 3 uses the fixed epoch-15 checkpoint.")
for key, expected in label_mappings.items():
    if key in checkpoint and json.dumps(checkpoint[key]) != json.dumps(expected):
        raise ValueError(f"Task 3 checkpoint has different {key}.")

state = checkpoint["model_state"]
if not torch.equal(state["allowed"].cpu(), allowed_pairs):
    raise ValueError("Task 3 checkpoint has a different set of allowed binary rules.")
supervised_grammar.load_state_dict(state)
supervised_grammar.eval()
record_checkpoint("task3", task3_path, checkpoint)
save_json("task3_mappings.json", label_mappings)
print(f"Task 3 checkpoint: {task3_path.name}, epoch {checkpoint['epoch']}")
del checkpoint, state

Task 3 checkpoint: task3_epoch15.pt, epoch 15


### CKY with the learned scores

For each rule and split, the score is left child + right child + rule. NumPy computes those candidates together. For each rule I keep its best split, then for each parent I keep its best rule. The backpointers recover the tree.

In [11]:
with torch.no_grad():
    root_log, binary_log, word_log = (scores.cpu() for scores in supervised_grammar())

labels = nonterminal_labels + preterminal_labels
label_id = {label: i for i, label in enumerate(labels)}
root_ids = np.array([label_id[label] for label in root_labels])
root_log_scores = np.array([float(score) for score in root_log])
word_log_scores = word_log.numpy()

# Keep a fixed rule order for ties.
rules_by_children = defaultdict(list)
for parent, left, right in sorted(observed_rules):
    rules_by_children[left, right].append(parent)
cky_rules = [
    (parent, left, right)
    for (left, right), parents in rules_by_children.items()
    for parent in parents
]
parents, left_children, right_children = np.array([
    [label_id[parent], label_id[left], label_id[right]] for parent, left, right in cky_rules
]).T
binary_log_scores = np.array([
    binary_log[nonterminal_id[parent], pair_id[left, right]].item()
    for parent, left, right in cky_rules
], dtype=np.float32)
rule_numbers = np.arange(len(cky_rules))

In [12]:
def parse_supervised(words, rule_scores=None):
    if rule_scores is None:
        rule_scores = binary_log_scores
    n = len(words)
    if n == 0:
        return None
    label_count = len(labels)
    chart = np.full((n, n + 1, label_count), -np.inf, dtype=np.float32)
    back_rule = np.full(chart.shape, -1, dtype=np.int32)
    back_split = np.full(chart.shape, -1, dtype=np.int32)

    word_ids = encode_words(words, supervised_vocab)
    for start, word_id in enumerate(word_ids):
        chart[start, start + 1, len(nonterminal_labels):] = word_log_scores[:, word_id]

    for width in range(2, n + 1):
        for start in range(n - width + 1):
            end = start + width
            splits = np.arange(start + 1, end)

            # One row per split, one column per rule.
            candidates = (
                chart[start, splits[:, None], left_children]
                + chart[splits[:, None], end, right_children]
                + rule_scores
            )
            best_split = candidates.argmax(axis=0)
            best_rule_score = candidates[best_split, rule_numbers]

            # Update the best score for each parent, even when its ID repeats.
            best_parent_score = chart[start, end]
            np.maximum.at(best_parent_score, parents, best_rule_score)
            winning = np.isfinite(best_rule_score) & (best_rule_score == best_parent_score[parents])
            best_rule = np.full(label_count, len(cky_rules))
            np.minimum.at(best_rule, parents[winning], rule_numbers[winning])
            valid = best_rule < len(cky_rules)
            back_rule[start, end, valid] = best_rule[valid]
            back_split[start, end, valid] = splits[best_split[best_rule[valid]]]

    full_scores = chart[0, n, root_ids] + root_log_scores
    best_root = full_scores.argmax()
    if not np.isfinite(full_scores[best_root]):
        return None

    def build_tree(start, end, label):
        if end - start == 1:
            return Tree(labels[label], [words[start]])
        rule = back_rule[start, end, label]
        split = back_split[start, end, label]
        return Tree(labels[label], [
            build_tree(start, split, left_children[rule]),
            build_tree(split, end, right_children[rule]),
        ])

    tree = build_tree(0, n, root_ids[best_root])
    tree.un_chomsky_normal_form()
    return tree

In [13]:
def evaluate_supervised(split, rule_scores=None, name="task3"):
    records = []
    for index, gold in enumerate(trees_by_split[split]):
        prediction = None
        if content_length(gold) >= 2:
            prediction = parse_supervised(gold.leaves(), rule_scores)
        records.append(score_sentence(split, index, gold, prediction))
    return save_evaluation(f"{name}_{split}", records), records


task3_validation, _ = evaluate_supervised("validation")
task3_test, task3_records = evaluate_supervised("test")
display(pd.DataFrame([
    score_row("Validation", task3_validation), score_row("Test", task3_test)
]).rename(columns={"Model": "Split"}).round(2))
show_examples(task3_records, "task3")

High F1 · test sentence 1 · F1 100.0%
Takuma Yamamoto , president of Fujitsu Ltd. , believes `` the ` money worship ' among young people ... caused the problem . ''
Gold:      (S
  (NP
    (NP (NNP Takuma) (NNP Yamamoto))
    (, ,)
    (NP (NP (NN president)) (PP (IN of) (NP (NNP Fujitsu) (NNP Ltd.))))
    (, ,))
  (VP
    (VBZ believes)
    (`` ``)
    (SBAR
      (S
        (NP
          (NP (DT the) (`` `) (NN money) (NN worship) ('' '))
          (PP (IN among) (NP (JJ young) (NNS people))))
        (: ...)
        (VP (VBD caused) (NP (DT the) (NN problem))))))
  (. .)
  ('' ''))
Predicted: (S
  (NP
    (NP (JJ Takuma) (NNS Yamamoto))
    (, ,)
    (NP (NP (NN president)) (PP (IN of) (NP (NNP Fujitsu) (NNP Ltd.))))
    (, ,))
  (VP
    (VBZ believes)
    (`` ``)
    (S
      (NP
        (NP (DT the) (`` `) (NN money) (NN worship) (POS '))
        (PP (IN among) (NP (JJ young) (NNS people)))
        (: ...))
      (VP (VBD caused) (NP (DT the) (NN problem)))
      (. .)
      ('' '

,Split,Scored,Complete parses,Precision (%),Recall (%),Sentence F1 (%),Corpus F1 (%)
0,Validation,130,130,68.50,65.01,72.23,66.71
1,Test,130,130,67.88,65.68,70.64,66.76


In [14]:
run_info.pop("task2_mode", None)
save_json("run_manifest.json", run_info)
(run_dir / "requirements_runtime.txt").write_text(
    "\n".join(f"{name}=={number}" for name, number in package_versions.items()) + "\n"
)
print(f"Results: {run_dir.resolve()}")

Results: /workspace/scratch/f479fa55acc2/submission_work/checks/task3/execution/results/20261107_125947_281910
